In [62]:
import os
import numpy as np
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [63]:
## Notes from pyracipe

# OUTPUT FILES FROM RUNNING RACIPE in the full mode

# cellcycle.states.txt - gene expression for the stable states
# cellcycle.limitcycles.txt - gene expression for the stable limit cycles
# cellcycle.summary.txt - summary of the number of states and limit cycles
# cellcycle.mpr.txt - MPR (maximum production rate) values of all the genes in each model
# cellcycle.dnr.txt - DNR (degradation rate) values of all the genes in each model
# cellcycle.tsh.txt - TSH (threshold) values used in all the interactions in each model
# cellcycle.hco.txt - HCO (Hill coefficient) values used in all the interactions in each model
# cellcycle.fch.txt - FCH (fold change) values in all the interactions in each model
# cellcycle.mpr.prs, cellcycle.dnr.prs, cellcycle.tsh.prs, cellcycle.hco.prs, and cellcycle.fch.prs stores the ranges for the MPR, DNR, TSH, HCO, and FCH, respectively.

In [64]:
def standardize_dataframe(df):
    # Initialize lists to store column means and standard deviations
    tmpMeans = []
    tmpSds = []
    
    # Copy the dataframe to avoid changing the original data
    standardized_df = df.copy()
    
    # Loop through each column in the DataFrame
    for column in df.columns:
        # Calculate mean and standard deviation for the column
        col_mean = df[column].mean()
        col_std = df[column].std()
        
        # Append the mean and std to the lists
        tmpMeans.append(col_mean)
        tmpSds.append(col_std)
        
        # Standardize the column
        standardized_df[column] = (df[column] - col_mean) / col_std
    
    return standardized_df, tmpMeans, tmpSds


In [65]:
# Using python racipe implementation, to calculate multi-stability

# Set up constants, filenames, etc
TOPO_NAME = "emt_ffctopo_72node_5k200ic"
PYRACIPE_DIR = "pyracipe/"+TOPO_NAME+"/"   

PYR_STATES = PYRACIPE_DIR+TOPO_NAME+".states.txt"
PYR_LIMITCYCLES = PYRACIPE_DIR+TOPO_NAME+".limitcycles.txt"
PYR_SUMMARY = PYRACIPE_DIR+TOPO_NAME+".summary.txt"

PYR_MPR = PYRACIPE_DIR+TOPO_NAME+".mpr.txt"
PYR_DNR = PYRACIPE_DIR+TOPO_NAME+".dnr.txt"
PYR_TSH = PYRACIPE_DIR+TOPO_NAME+".tsh.txt"
PYR_HCO = PYRACIPE_DIR+TOPO_NAME+".hco.txt"
PYR_FCH = PYRACIPE_DIR+TOPO_NAME+".fch.txt"



In [66]:
# Also read in second simulation part
# Set up constants, filenames, etc
TOPO_NAME_2 = "emt_ffctopo_72node_5k200ic_2025"
PYRACIPE_2_DIR = "pyracipe/"+TOPO_NAME_2+"/"   

PYR_STATES_2 = PYRACIPE_2_DIR+TOPO_NAME_2+".states.txt"
PYR_LIMITCYCLES_2 = PYRACIPE_2_DIR+TOPO_NAME_2+".limitcycles.txt"
PYR_SUMMARY_2 = PYRACIPE_2_DIR+TOPO_NAME_2+".summary.txt"

PYR_MPR_2 = PYRACIPE_2_DIR+TOPO_NAME_2+".mpr.txt"
PYR_DNR_2 = PYRACIPE_2_DIR+TOPO_NAME_2+".dnr.txt"
PYR_TSH_2 = PYRACIPE_2_DIR+TOPO_NAME_2+".tsh.txt"
PYR_HCO_2 = PYRACIPE_2_DIR+TOPO_NAME_2+".hco.txt"
PYR_FCH_2 = PYRACIPE_2_DIR+TOPO_NAME_2+".fch.txt"



In [82]:
# Read the tsv file into a pandas DataFrame
df = pd.read_csv(PYR_STATES, sep="\t")
df2 = pd.read_csv(PYR_STATES_2, sep="\t")
df2.MODEL_NO = df2.MODEL_NO + 5000

df_comb = pd.concat([df, df2])
df_comb.index = list(range(1, df_comb.shape[0]+1, 1))
                  
# Separate metadata and data
metadata = df_comb[["MODEL_NO", "NO_STATES", "STATE_NO"]]
data = df_comb.drop(columns=["MODEL_NO", "NO_STATES", "STATE_NO","Unnamed: 75"])


data

,ILK,AKT,PI3K,AXIN2,TCFLEF,CD44,TGFR,CDC42,CHD1L,NOTCHic,...,SNAI2,cMet,TGF,ZEB2,HGF,SUFU,RKIP,Patched,TrCP,miR200
1,6.445005,7.652006,1.430474,-1.969997,6.540261,2.671531,8.093070,7.215658,8.008352,-2.052664,...,6.122878,-0.405505,7.902051,-2.287371,5.998151,5.032750,-1.977728,-1.488658,4.265334,-5.362880
2,6.445005,7.652006,1.430474,2.735968,6.540261,2.671531,8.093070,7.215658,8.008352,-2.052664,...,6.122878,-0.405505,7.902051,-2.287371,5.998151,5.032750,-1.977728,-1.488658,4.265334,-5.362880
3,2.252713,3.539781,2.871819,0.868132,-0.246457,0.740719,1.264631,7.662651,7.079389,0.547476,...,-10.700186,-2.890244,-11.124932,-17.915876,6.090471,6.360218,2.533632,7.453136,7.193103,5.771310
4,5.972062,2.862781,4.905702,-1.028491,5.662110,5.956306,6.679732,1.974690,3.198011,7.264843,...,6.087553,4.983416,5.397828,8.637980,9.393755,2.808514,-1.369767,2.004415,4.876280,-7.833845
5,1.218401,-7.469699,-1.413864,-3.077925,0.114429,3.616092,0.568514,-4.123414,3.198011,0.941667,...,-8.974989,1.484170,-6.820635,1.697835,9.393755,2.808750,6.790120,2.249453,5.087860,3.553738
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20334,4.877881,3.919712,5.742835,6.488336,8.711499,6.581514,5.940000,6.696724,8.088891,4.534538,...,7.012729,4.986464,5.710630,-5.756606,6.419481,4.191561,-3.061974,1.926017,0.610848,2.080172
20335,4.877892,3.919737,5.742835,6.488336,8.711499,6.581514,5.940000,6.696724,8.088891,4.534538,...,7.011094,4.986464,5.710636,-5.691248,6.419481,4.191561,-3.090555,1.925998,0.610847,2.052109
20336,-0.916379,0.682392,1.604733,6.488317,8.586163,6.581514,0.196865,3.264918,8.088891,-1.745726,...,0.188642,4.986464,-1.431994,-12.819283,6.419481,8.832210,6.758616,6.508584,2.034212,4.783940
20337,4.874735,3.912219,5.742835,6.488336,8.711498,6.581514,5.937368,6.696146,8.088891,4.534538,...,6.748065,4.986464,4.116631,-7.394307,6.419481,8.831602,0.066333,6.400709,0.611067,3.398452


In [83]:
pd.DataFrame(data).to_csv(PYRACIPE_DIR+TOPO_NAME+"_pyracipe_states_unStdized_2025_combined.csv")

In [84]:
# Standardize data
data, tmpMeans, tmpSds = standardize_dataframe(data)

data

,ILK,AKT,PI3K,AXIN2,TCFLEF,CD44,TGFR,CDC42,CHD1L,NOTCHic,...,SNAI2,cMet,TGF,ZEB2,HGF,SUFU,RKIP,Patched,TrCP,miR200
1,1.395949,1.961732,0.148025,-0.509717,0.996389,-0.368335,1.913942,1.713951,1.074242,-1.627179,...,1.746257,-0.778987,2.140852,0.383950,-0.210624,0.163440,-1.702164,-2.713489,-0.283935,-1.504183
2,1.395949,1.961732,0.148025,0.484907,0.996389,-0.368335,1.913942,1.713951,1.074242,-1.627179,...,1.746257,-0.778987,2.140852,0.383950,-0.210624,0.163440,-1.702164,-2.713489,-0.283935,-1.504183
3,-0.095368,0.944442,0.532633,0.090132,-1.270918,-1.045095,-0.445035,1.844991,0.470500,-0.705919,...,-0.416737,-1.517404,-0.912288,-2.398968,-0.150886,0.673317,-0.472877,0.891474,0.880575,0.509869
4,1.227710,0.776964,1.075354,-0.310726,0.703017,0.782997,1.425686,0.177510,-2.052046,1.674122,...,1.741715,0.822499,1.739015,2.329393,1.986557,-0.690882,-1.536503,-1.305226,-0.040933,-1.951153
5,-0.463302,-1.779105,-0.610957,-0.743882,-1.150353,-0.037261,-0.685518,-1.610209,-2.052046,-0.566253,...,-0.194923,-0.217411,-0.221604,1.093583,1.986557,-0.690792,0.686962,-1.206437,0.043222,0.108735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20334,0.838479,1.038430,1.298734,1.277984,1.721757,1.002136,1.170135,1.561820,1.126585,0.706743,...,1.860667,0.823405,1.789208,-0.233805,0.062005,-0.159658,-1.997607,-1.336833,-1.737493,-0.157817
20335,0.838482,1.038436,1.298734,1.277984,1.721757,1.002136,1.170135,1.561820,1.126585,0.706743,...,1.860457,0.823405,1.789209,-0.222167,0.062005,-0.159658,-2.005395,-1.336840,-1.737493,-0.162893
20336,-1.222703,0.237575,0.194524,1.277980,1.679884,1.002136,-0.813909,0.555753,1.126585,-1.518427,...,0.983274,0.823405,0.643077,-1.491434,0.062005,1.622802,0.678377,0.510670,-1.171355,0.331265
20337,0.837359,1.036576,1.298734,1.277984,1.721756,1.002136,1.169226,1.561651,1.126585,0.706743,...,1.826639,0.823405,1.533429,-0.525426,0.062005,1.622568,-1.145184,0.467179,-1.737406,0.080646


In [86]:
summary = pd.read_csv(PYR_SUMMARY, sep="\t")
summary_p2 = pd.read_csv(PYR_SUMMARY_2, sep="\t")
summary_comb = pd.concat([summary, summary_p2])
summary_comb.MODEL_NO = list(range(1, summary_comb.shape[0]+1, 1))
summary_comb.index = list(range(1, summary_comb.shape[0]+1, 1))
summary_comb

,MODEL_NO,NO_STATES,NO_LIMITCYCLES,CAL_TIME_FOR_LCS,Unnamed: 4
1,1,2,0,1.165098,NaN
2,2,1,0,0.000048,NaN
3,3,2,0,0.000021,NaN
4,4,3,0,0.000037,NaN
5,5,1,0,2.596869,NaN
...,...,...,...,...,...
9996,9996,1,0,0.000022,NaN
9997,9997,3,0,0.000028,NaN
9998,9998,5,0,0.000020,NaN
9999,9999,5,0,2.126075,NaN


In [87]:
summary_comb["NO_STATES"].mean()

2.0338

In [88]:
pd.DataFrame(data).to_csv(PYRACIPE_DIR+TOPO_NAME+"_pyracipe_states_2025_combined.csv")

pd.DataFrame(metadata).to_csv(PYRACIPE_DIR+TOPO_NAME+"_pyracipe_metadata_2025_combined.csv")

pd.DataFrame(summary_comb).to_csv(PYRACIPE_DIR+TOPO_NAME+"_pyracipe_summary_2025_combined.csv")


In [89]:
param_table = pd.read_csv(PYR_MPR, sep="\t")
param_table

,MODEL_NO,ILK,AKT,PI3K,AXIN2,TCFLEF,CD44,TGFR,CDC42,CHD1L,...,cMet,TGF,ZEB2,HGF,SUFU,RKIP,Patched,TrCP,miR200,Unnamed: 73
0,1,38.210426,97.994107,34.897698,4.766870,72.889031,6.845366,67.547208,46.494803,64.284016,...,93.616654,55.031871,16.801109,23.165515,52.253557,42.975158,13.713991,48.517301,78.968288,NaN
1,2,30.507560,73.467522,36.512919,40.904962,8.670925,75.050259,92.930415,68.874372,92.607318,...,6.313491,81.273237,10.890025,34.555782,80.104921,70.852273,99.313831,57.239438,38.711020,NaN
2,3,52.081537,7.911173,25.543619,55.188903,66.427970,49.278170,86.287125,22.630710,4.895959,...,68.200252,28.085001,77.196004,86.655236,69.193398,72.404548,6.549938,28.319534,88.832917,NaN
3,4,93.612451,89.381227,23.789223,95.057287,31.998566,43.083520,97.092922,34.342190,80.920348,...,91.720453,9.118675,66.560910,29.917724,97.791684,69.876251,93.032779,35.570691,17.376072,NaN
4,5,99.721173,26.185008,70.274044,98.058103,60.708041,96.430902,3.512685,78.983347,21.433126,...,74.043869,94.213475,76.611068,62.623978,98.485755,24.938639,18.895451,1.824402,19.128661,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,14.061450,66.359577,25.942665,68.783587,28.796020,37.464913,4.997903,5.926815,32.523937,...,74.685744,92.269541,73.560764,56.868018,50.850671,64.479165,37.342324,48.995351,47.305996,NaN
4996,4997,32.602353,91.767100,1.335289,18.764347,55.999556,58.311422,21.051643,62.460705,70.241375,...,50.112852,3.483965,31.382279,3.352300,76.146957,70.619017,52.111167,45.179696,9.420156,NaN
4997,4998,78.255612,57.406480,50.199684,91.243111,65.743967,84.164032,28.474811,71.174786,48.012031,...,16.439344,78.919116,74.752700,37.526686,71.508751,96.521672,55.598492,78.152872,82.891226,NaN
4998,4999,39.546342,98.051229,43.920874,62.508659,71.932220,83.653371,64.125802,22.510078,33.314281,...,8.053869,40.100547,6.702801,44.922218,55.277487,30.772896,34.493827,10.798665,93.378932,NaN


In [90]:
# Merge and save parameters
# Names of the parameter tables and corresponding parameter labels
tables = [(PYR_MPR, PYR_MPR_2), (PYR_DNR, PYR_DNR_2), (PYR_TSH, PYR_TSH_2), (PYR_HCO, PYR_HCO_2), (PYR_FCH, PYR_FCH_2)]
params = ['MPR', 'DNR', 'TSH', 'HCO', 'FCH']

# Import tables and merge
dfs = []
for (table, table2), param in zip(tables, params):
    # Import table
    df = pd.read_csv(table, sep="\t")
    df2 = pd.read_csv(table2, sep="\t")

    df_comb = pd.concat([df, df2])
    df_comb.MODEL_NO = list(range(1, df_comb.shape[0]+1, 1))
    df_comb.index = list(range(1, df_comb.shape[0]+1, 1))

    # Trim off extra column
    if param in ['MPR','DNR']:
        # Drop any extra columns beyond the 73rd
        df_comb = df_comb.iloc[:, :73]
    else:
        df_comb = df_comb.iloc[:, :143]
    
    # Rename columns
    new_columns = {col: f"{param}_{col}" for col in df_comb.columns if col != "MODEL_NO"}
    df_comb.rename(columns=new_columns, inplace=True)
    
    # Append to list
    dfs.append(df_comb)

# Merge the tables based on MODEL_NO
merged_df = dfs[0]
for df in dfs[1:]:
    merged_df = pd.merge(merged_df, df, on="MODEL_NO")


In [91]:
merged_df.shape

(10000, 571)

In [92]:
merged_df

,MODEL_NO,MPR_ILK,MPR_AKT,MPR_PI3K,MPR_AXIN2,MPR_TCFLEF,MPR_CD44,MPR_TGFR,MPR_CDC42,MPR_CHD1L,...,FCH_miR200-ZEB2,FCH_SNAI1-miR200,FCH_ZEB1-miR200,FCH_ZEB2-miR200,FCH_cateninnuc-cateninmemb,FCH_Ecadherin-cateninnuc,FCH_cateninmemb-cateninnuc,FCH_Destcompl-cateninnuc,FCH_SUFU-cateninnuc,FCH_Csn-TrCP
0,1,38.210426,97.994107,34.897698,4.766870,72.889031,6.845366,67.547208,46.494803,64.284016,...,0.073651,0.016504,0.011391,0.029485,0.019119,0.119418,0.017140,0.028663,0.214029,0.051437
1,2,30.507560,73.467522,36.512919,40.904962,8.670925,75.050259,92.930415,68.874372,92.607318,...,0.012430,0.010256,0.030545,0.010373,0.017970,0.010715,0.016830,0.014239,0.028442,0.020913
2,3,52.081537,7.911173,25.543619,55.188903,66.427970,49.278170,86.287125,22.630710,4.895959,...,0.078592,0.017494,0.024236,0.021469,0.013966,0.034104,0.136323,0.010834,0.059600,0.019551
3,4,93.612451,89.381227,23.789223,95.057287,31.998566,43.083520,97.092922,34.342190,80.920348,...,0.010810,0.012552,0.024094,0.026996,0.016754,0.018993,0.012807,0.019559,0.011395,0.030951
4,5,99.721173,26.185008,70.274044,98.058103,60.708041,96.430902,3.512685,78.983347,21.433126,...,0.023904,0.014946,0.180352,0.043051,0.034213,0.015022,0.025284,0.014189,0.037514,0.015182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,14.061450,66.359577,25.942665,68.783587,28.796020,37.464913,4.997903,5.926815,32.523937,...,0.015579,0.586674,0.075664,0.012459,0.047009,0.057900,0.041333,0.052312,0.060184,0.033939
9996,9997,32.602353,91.767100,1.335289,18.764347,55.999556,58.311422,21.051643,62.460705,70.241375,...,0.021702,0.016043,0.010067,0.266149,0.017842,0.102902,0.014558,0.034161,0.023922,0.013116
9997,9998,78.255612,57.406480,50.199684,91.243111,65.743967,84.164032,28.474811,71.174786,48.012031,...,0.011559,0.020037,0.061020,0.035806,0.013559,0.043034,0.398702,0.694096,0.203581,0.034485
9998,9999,39.546342,98.051229,43.920874,62.508659,71.932220,83.653371,64.125802,22.510078,33.314281,...,0.010332,0.018978,0.066694,0.013637,0.027761,0.057358,0.014535,0.113141,0.050886,0.542205


In [93]:
#merged_df
pd.DataFrame(merged_df).to_csv(PYRACIPE_DIR+TOPO_NAME+"_pyracipe_params_2025_combined.csv")